[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/9-evals.ipynb)

## Colab setup

Run the next cell first on Google Colab. It sparse-checkouts this public repo and installs packages only when the kernel is Colab. Local Jupyter and VS Code skip that work.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) or in an environment cell: `os.environ["OPENAI_API_KEY"] = "sk-..."`. Do that before the lesson setup cell. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

In [ ]:
# Colab setup. Run this cell before the other code cells.
# A fresh Colab runtime cannot import tutorial.common yet, so this cell
# sparse-checkouts the repo first. Local Jupyter and VS Code skip that.
#
# An API key is required. On Google Colab, set OPENAI_API_KEY in one of these ways:
#   * Secrets (the key icon): a secret named OPENAI_API_KEY
#   * an environment cell: os.environ["OPENAI_API_KEY"] = "sk-..."
# Locally, put the key in the repository-root .env.
# Do not commit a key. This cell does not print the value.
# Edits you make in Colab stay in the session. They do not push to GitHub.

import sys
from pathlib import Path


def _on_colab():
    try:
        import google.colab
    except ImportError:
        return False
    return Path("/content").is_dir() and google.colab is not None


def _use(root):
    if not (Path(root) / "tutorial" / "common" / "colab.py").is_file():
        return False
    text = str(root)
    if text not in sys.path:
        sys.path.insert(0, text)
    return True


ready = False
if _on_colab():
    repo = Path("/content/book-agents")
    if not _use(repo):
        import shutil
        import subprocess

        if repo.exists() and not (repo / ".git").exists():
            raise RuntimeError(
                str(repo) + " exists but is not a git checkout. "
                "Move that folder aside and run this cell again."
            )
        if repo.exists():
            shutil.rmtree(repo)
        url = "https://github.com/junlinghu/book-agents.git"
        try:
            subprocess.check_call([
                "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                url, str(repo),
            ])
            subprocess.check_call([
                "git", "-C", str(repo), "sparse-checkout", "set", "tutorial",
            ])
        except subprocess.CalledProcessError:
            if repo.exists():
                shutil.rmtree(repo)
            subprocess.check_call(["git", "clone", "--depth", "1", url, str(repo)])
        if not _use(repo):
            raise RuntimeError(
                "Colab setup could not find tutorial/common/colab.py after cloning."
            )
    ready = True
else:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if _use(candidate):
            ready = True
            break
    if not ready:
        print("Not Colab. Skipped clone and pip install.")

if ready:
    from tutorial.common.colab import setup_colab

    setup_colab()


# 9. Evals

You already know two answers the concierge must get right, and one answer it once got wrong. That is an eval set.

`grade_answer` checks phrases and whether a tool ran. The opened-coffee case uses `get_shop_fact`. The milk case uses `read_shop_file`. The Wi-Fi case is a canned bad answer. It does not call the model. It stays in the set because it fails. Those two tools and the loop are imported above.

Shared helpers this lesson needs are imported, not copied from earlier notebooks. You can run this file by itself.

## Setup

This notebook calls the OpenAI Chat Completions API. `OPENAI_API_KEY` is required. Locally the key is loaded from the repository-root `.env` by `tutorial/common/client.py`. On Colab, the setup cell above reads a secret of the same name. The value is not printed. A missing key stops the run.

Companion notes: `9-evals.md`.

In [ ]:
import sys
from pathlib import Path


def find_root():
    """Repo root, whether the kernel started here or in tutorial."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        marker = candidate / "tutorial" / "common" / "client.py"
        if marker.is_file():
            return candidate
    raise RuntimeError(
        "Open this notebook inside the book-agents repository. "
        "The setup cell looks for tutorial/common/client.py."
    )


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.common.client import describe_runtime
from tutorial.runtime import chat, require_key

LESSON = '9-evals'
LESSON_NUMBER = 9
require_key()
print("tutorial:", LESSON)
print(describe_runtime())


## Shared pieces

These imports are the earlier pieces this lesson uses. They come from `tutorial.common` and `tutorial.runtime`. You do not need to run the previous notebook first.


In [ ]:
from tutorial.common.facts import get_shop_fact  # registers the return-rule tool
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.shelf import read_shop_file, reset_db


## Tutorial 9 — graders

A tiny grader over the answer and the tool log. Keep a failure in the set.

In [ ]:
def grade_answer(text, tool_log, contains, excludes, tool_name=None):
    """Small grader. A failure is a list of reasons. An empty list is a pass."""
    failures = []
    lowered = (text or "").lower()
    for needle in contains:
        if needle.lower() not in lowered:
            failures.append("missing " + needle)
    for banned in excludes:
        if banned.lower() in lowered:
            failures.append("should not say " + banned)
    if tool_name and not any(tool_name in line for line in tool_log):
        failures.append("did not use " + tool_name)
    return failures


def print_grade(case_id, failures):
    if failures:
        print("FAIL", case_id + ":", "; ".join(failures))
    else:
        print("PASS", case_id)
    return not failures


## Run

`system_text()` mentions only the tools imported above. Run this cell after the ones above. Each model turn calls the Chat Completions API. A check prints a warning when the wording differs from the expected trace, and the notebook continues.

In [ ]:
reset_db()
cases = [
    {
        "id": "opened-return",
        "question": "I opened the house coffee. Can I return it?",
        "contains": ["final sale"],
        "excludes": ["full cash refund"],
        "tool": "get_shop_fact",
    },
    {
        "id": "ship-milk",
        "question": "Please ship a gallon of whole milk to Ohio.",
        "contains": ["does not ship"],
        "excludes": ["tracking number"],
        "tool": "read_shop_file",
    },
]
passed = 0
for case in cases:
    print("===", case["id"], "===")
    result = run_agent(case["question"], system_text())
    print("ANSWER:", result["text"])
    failures = grade_answer(
        result["text"],
        result["tool_log"],
        case["contains"],
        case["excludes"],
        case["tool"],
    )
    if print_grade(case["id"], failures):
        passed += 1

print("=== invented-wifi (a failure kept from an earlier bad answer) ===")
canned = "The Wi-Fi password is hearth123."
print("CANNED ANSWER:", canned)
failures = grade_answer(canned, [], [], ["hearth123"], "read_shop_file")
if print_grade("invented-wifi", failures):
    passed += 1
else:
    check(True, "the canned failure is supposed to fail the grader")
print("scored_passes:", passed, "of 3 (the canned row is in the set because it fails)")
check(passed == 2, "the two tool-using cases pass")
